In [ ]:
# =============================================================================
# Group Demographic Statistics and Standardized Scores
#
# PURPOSE:
# This notebook summarizes participant demographics and standardized assessment
# scores for the CWS and CWNS groups.
#
# INPUT:
# - CWNS_CWS_all_demo_and_score_details.csv
#
# OUTPUT:
# - Descriptive statistics for participant characteristics
# - Group comparisons for demographic and score variables
# - Summary values used for Table 1
#
# NOTE:
# This notebook is part of the analysis stage and should be run after the
# full participant-level dataset has been created.
# =============================================================================

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import median_test

In [ ]:
# Project directory structure
#
# ATAS_CWNS_CWS_Project/        ← THIS is base_dir
# │
# ├── Audio_files_clipped/      ← create this folder; clipped audio files stored here
# ├── Visualize/                ← create this folder; .wav audio files for visualization/analysis stored here
# ├── Stat_csv_files/           ← input CSV files and summary statistics CSV files stored here
# ├── Individual_OutputCSV_Files/ ← individual output CSV files stored here
# │
# ├── Notebooks/                ← notebooks stored here
# │   ├── ATAS_Visualization/   ← visualization notebook
# │   └── ML_Analysis/          ← machine learning analysis notebooks
# │
# └── Scripts/                  ← function notebooks
#
base_dir = '..../ATAS_CWNS_CWS_Project'  # Change to required project directory

csv_file = base_dir + "/Stat_csv_files/CWNS_CWS_all_demo_and_score_details.csv" 


In [ ]:
## Descriptive Statistics for Participant Characterists and Standardized Measures (Table 1)

# ======================================================
# Load data
# ======================================================
df = pd.read_csv(csv_file)

# Clean group labels
df["Group"] = df["Group"].astype(str).str.strip()

# ======================================================
# Helper functions
# ======================================================
def desc(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return {"n": 0, "mean": np.nan, "sd": np.nan, "min": np.nan, "max": np.nan}
    return {
        "n": len(s),
        "mean": s.mean(),
        "sd": s.std(ddof=1) if len(s) > 1 else np.nan,
        "min": s.min(),
        "max": s.max()
    }

def welch_test(x, y):
    x = pd.to_numeric(x, errors="coerce").dropna()
    y = pd.to_numeric(y, errors="coerce").dropna()
    t, p = stats.ttest_ind(x, y, equal_var=False)
    return t, p

def cohen_d(x, y):
    x = pd.to_numeric(x, errors="coerce").dropna()
    y = pd.to_numeric(y, errors="coerce").dropna()

    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan

    sx, sy = x.std(ddof=1), y.std(ddof=1)
    pooled_sd = np.sqrt(((nx - 1) * sx**2 + (ny - 1) * sy**2) / (nx + ny - 2))

    if pooled_sd == 0:
        return np.nan

    return (x.mean() - y.mean()) / pooled_sd

def hedges_g(x, y):
    x = pd.to_numeric(x, errors="coerce").dropna()
    y = pd.to_numeric(y, errors="coerce").dropna()

    d = cohen_d(x, y)
    if pd.isna(d):
        return np.nan

    nx, ny = len(x), len(y)
    J = 1 - (3 / (4 * (nx + ny) - 9))
    return d * J

# ======================================================
# Define variables
# ======================================================
all_desc_vars = [
    "Age", "TONI", "IQ", "Final_all_IQ",
    "PPVT", "EVT", "GFTA", "ses_edu",
    "SSI", "%SS"
]

# Variables to compare across groups
# SSI and %SS will only be tested if values are real
test_vars = [
    "Age", "TONI", "IQ", "Final_all_IQ",
    "PPVT", "EVT", "GFTA", "ses_edu",
    "SSI", "%SS"
]

# ======================================================
# Identify groups
# ======================================================
groups = [g for g in df["Group"].dropna().unique()]

if len(groups) != 2:
    raise ValueError(f"Expected exactly 2 groups, found {len(groups)}: {groups}")

g1, g2 = groups

# ======================================================
# DESCRIPTIVE STATISTICS
# ======================================================
summary_rows = []

for var in all_desc_vars:
    if var not in df.columns:
        continue

    # For SSI and %SS, exclude rows where values are structurally missing
    # Example: CWNS may have NaN by design
    if var in ["SSI", "%SS"]:
        df_var = df.loc[df[var].notna()].copy()
    else:
        df_var = df.copy()

    # Overall
    overall_stats = desc(df_var[var])
    summary_rows.append({
        "Variable": var,
        "Group": "Overall" if var not in ["SSI", "%SS"] else "Available cases only",
        **overall_stats,
        "missing_n": df[var].isna().sum(),
        "missing_%": df[var].isna().mean() * 100
    })

    # By group
    for g in [g1, g2]:
        sub = df.loc[df["Group"] == g, var]
        group_stats = desc(sub)

        summary_rows.append({
            "Variable": var,
            "Group": g,
            **group_stats,
            "missing_n": sub.isna().sum(),
            "missing_%": sub.isna().mean() * 100
        })

summary_df = pd.DataFrame(summary_rows)

# ======================================================
# GROUP COMPARISONS (Welch t-tests)
# ======================================================
test_rows = []

for var in test_vars:
    if var not in df.columns:
        continue

    x = pd.to_numeric(df.loc[df["Group"] == g1, var], errors="coerce")
    y = pd.to_numeric(df.loc[df["Group"] == g2, var], errors="coerce")

    n1 = x.notna().sum()
    n2 = y.notna().sum()

    # Skip if too few observations
    if n1 < 2 or n2 < 2:
        continue

    t, p = welch_test(x, y)

    test_rows.append({
        "Variable": var,
        "Group 1": g1,
        "Group 2": g2,
        "n1": n1,
        "n2": n2,
        "Mean 1": x.mean(),
        "SD 1": x.std(ddof=1),
        "Mean 2": y.mean(),
        "SD 2": y.std(ddof=1),
        "t (Welch)": t,
        "p-value": p
    })

tests_df = pd.DataFrame(test_rows)

# ======================================================
# DISPLAY RESULTS
# ======================================================
print("\n=== DESCRIPTIVE STATISTICS ===")
display(summary_df.round(2))

print("\n=== GROUP COMPARISONS (Welch t-tests) ===")
display(tests_df.round(3))

# ======================================================
# SAVE OUTPUT FILES
# ======================================================
# summary_df.to_csv("summary_tables.csv", index=False)
# tests_df.to_csv("group_tests.csv", index=False)

print("\nFiles ready to save:")
print(" - summary_tables.csv")
print(" - group_tests.csv")

In [ ]:
## Testing group difference for SES (Hollinghead) scores

# Load data
df = pd.read_csv(csv_file)

# Drop missing values
df = df[['Group', 'ses_edu']].dropna()

# Split groups
cw_ns = df[df['Group'] == 'CWNS']['ses_edu']
cws = df[df['Group'] == 'CWS']['ses_edu']

# Run Mood's median test
stat, p_value, median, table = median_test(cw_ns, cws)

print(f"Overall median = {median}")
print(f"Test statistic = {stat}")
print(f"p-value = {p_value}")

if p_value < 0.05:
    print("Result: Significant difference (p < 0.05)")
else:
    print("Result: Not significant (p ≥ 0.05)")

print("\nContingency table (below vs above median):")
print(table)